In [1]:
import napari
import skimage as ski
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import glob
from sam2.build_sam import build_sam2_video_predictor

In [2]:
viewer = napari.Viewer()

In [3]:
def keep_largest(current_mask):
    labels = ski.measure.label(current_mask)
    props = ski.measure.regionprops(labels)
    largest_area = 0
    largest_label = 0
    for prop in props:
        if prop.area > largest_area:
            largest_area = prop.area
            largest_label = prop.label
    rtn = (labels == largest_label).astype(int)
    return rtn

def is_legit_shape(shape):
    area = np.sqrt(np.sum((shape.max(axis=0) - shape.min(axis=0))**2))
    if area > 10:
        return True
    return False
dummy_shape = np.array([[0,0,0], [0,0,1], [0, 1, 1], [0,1,0]])

# Load image, choose channel

In [4]:
#viewer.add_image(ski.data.cells3d(), name='cells3d', channel_axis=1)
#img = ski.io.imread('S:/micro/mg2/ks2438/mel/20210528_MER_IMARE-105429_nem_plan_150b/cjw2/stack_ali_zcorrected_bin16_rotated.tif')
#img = ski.io.imread('U:/smc/public/SMC/NuclearSegmentation3D/Datasets/NSIN/other_sets_unlabeled/planaria_40X/ex3/ex3.tif')
#img = ski.io.imread('S:/micro/kk2853/Grace/smc/MicroCT/CroppedB.tif')
#img = ski.io.imread('S:/micro/kk2853/Grace/smc/MicroCT/ImgB/examples_ds4/A.tif')
#img = ski.io.imread('S:/micro/jeg/amg/UBF_experiment/expansion_samples/UBF_Nucleolin_Expansion.lif - Old_follicle10_intensity_488.20_561-1.3_cropped.tif')[:,:,:,0]

#img = ski.io.imread('S:/micro/mg2/ak2718/20250418_DragonFly_mck/2025-04-18/ncol5_488_ncol1_546_dapi_ncol4_647_s01.tif')
#fname = 'S:/micro/mg2/ak2718/20250418_DragonFly_mck/2025-04-18/ncol5_488_ncol1_546_dapi_ncol4_647_s01.tif'
fname = 'S:/micro/mg2/ak2718/20250418_DragonFly_mck/2025-04-18/ncol5_488_ncol1_546_dapi_ncol4_647_s06.tif'
img = ski.io.imread(fname)
img = img[-1,:,:,:]
#img = ski.transform.rescale(img, (1,0.25,0.25), preserve_range=True)
z = np.arange(0, img.shape[0])
z_scaler = np.exp(-z/100)
clipped = np.clip(img.astype(np.float32)-92.0,0.0, np.inf)
z_corrected = clipped/z_scaler[:, np.newaxis, np.newaxis]
z_corrected = ski.filters.gaussian(z_corrected, sigma=(3, 10, 10), preserve_range=True)

In [5]:
viewer.add_image(z_corrected, name='img')

<Image layer 'img' at 0x1bc56ff3410>

tmp = viewer.layers['master_labels'].data
ski.io.imsave('U:/smc/public/SMC/NuclearSegmentation3D/Datasets/NSIN/other_sets_unlabeled/planaria_40X/ex2/ex2_gt.tif', tmp)

In [6]:
img_to_process = np.sqrt(viewer.layers[0].data)

mlabels = viewer.layers['master_labels'].data
ski.io.imsave('S:/micro/kk2853/Grace/smc/MicroCT/ImgB/examples_ds4/A_mask.tif', mlabels)

# Once image chosen, run the rest, draw a box(s) with rectangle tool, press Shift-S to segment

In [7]:
def delete_tmp_files():
    tmp_dir = 'tmp/'
    if not os.path.exists(tmp_dir):
        os.makedirs(tmp_dir)
    files = glob.glob('tmp/*')
    for f in files:
        try:
            os.remove(f)
        except OSError as e:
            print(f"Error: {f} : {e.strerror}")

# Convert single channel 8 bit jpeg into 3 channel 8 bit jpeg
def convert_to_rgb(cimg):
    cimg = (cimg - cimg.min()) / (cimg.max() - cimg.min()) * 255
    for i, img in enumerate(cimg):
        img = Image.fromarray(img)
        img = img.convert("RGB")
        img.save(f'tmp/{i:04d}.jpg', quality=100)


In [8]:
delete_tmp_files()
convert_to_rgb(img_to_process)

In [9]:
device = torch.device("cuda")
if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

# Video with Box

In [10]:
sam2_checkpoint = "checkpoints/sam2_hiera_base_plus.pt"
model_cfg = "sam2_hiera_b+.yaml"

predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)

In [11]:
inference_state = predictor.init_state(video_path='tmp/')

fnames = glob.glob('tmp/' + '*.jpg')
fnames.sort()
image = np.array([ski.io.imread(f) for f in fnames])
viewer.add_shapes([dummy_shape], ndim=3)

frame loading (JPEG): 100%|██████████| 80/80 [00:06<00:00, 12.91it/s]
D:\segment-anything-2\sam2\modeling\backbones\hieradet.py:68: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  x = F.scaled_dot_product_attention(


<Shapes layer 'Shapes' at 0x1bc810b64b0>

In [12]:
def cleanup_mask(label_img, label, start_frame, cutoff=0.8):
    mask = (label_img == label).astype(int)
    bad_down = False
    bad_up = False
    for idx in range(start_frame-1, 0, -1):
        old_mask = mask[idx+1]
        new_mask = mask[idx]

        if np.sum(new_mask * old_mask) / np.sum(new_mask) < cutoff:
            label_img[idx] = label_img[idx] * 0
            bad_down = True
        if bad_down:
            label_img[idx] = label_img[idx] * 0
    
    for idx in range(start_frame+1, label_img.shape[0]):
        old_mask = mask[idx-1]
        new_mask = mask[idx]

        if np.sum(new_mask * old_mask) / np.sum(new_mask) < cutoff:
            label_img[idx] = label_img[idx] * 0
            bad_up = True
        if bad_up:
            label_img[idx] = label_img[idx] * 0

    return label_img

In [13]:
@viewer.bind_key('Shift-D', overwrite=True)
def process_image(viewer):
    #max_distance = 9
    max_distance = 90000
    global predictor
    global inference_state
    global image
    shapes = viewer.layers['Shapes'].data[1:]
    number_objects = len(shapes)
    predictor.reset_state(inference_state)

    for idx, shape in enumerate(shapes):
        rect = shape
        if not is_legit_shape(rect):
            print('Bad shape')
            continue
        box = np.array([np.min(rect[:,2]), np.min(rect[:,1]), np.max(rect[:,2]), np.max(rect[:,1])]).astype(int)
        ann_frame_idx = int(rect[0][0])
        ann_obj_id = idx+1

        
        _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=ann_frame_idx,
        obj_id=ann_obj_id,
        box=box,
        )

        # Could display all of these as we go, but won't bother for now

    # Propagate all of them backwards and forwards

    # Forward
    video_segments = {}  # video_segments contains the per-frame segmentation results
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
        video_segments[out_frame_idx] = {
            out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
            for i, out_obj_id in enumerate(out_obj_ids)
        }

    full_mask = np.zeros_like(image[:,:,:,0])
    for frame_idx, segmentations in video_segments.items():
        for obj in np.arange(1, number_objects+1):
            cur_mask = segmentations[obj][0]
            if np.abs(frame_idx-ann_frame_idx) < max_distance:
                full_mask[frame_idx] = full_mask[frame_idx] + cur_mask*obj

    # Reverse
    video_segments = {}  # video_segments contains the per-frame segmentation results
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state, reverse=True):
        video_segments[out_frame_idx] = {
            out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
            for i, out_obj_id in enumerate(out_obj_ids)
        }

    for frame_idx, segmentations in video_segments.items():
        for obj in np.arange(1, number_objects+1):
            if frame_idx == int(rect[0][0]):
                continue
            cur_mask = segmentations[obj][0]
            if np.abs(frame_idx-ann_frame_idx) < max_distance:
                full_mask[frame_idx] = full_mask[frame_idx] + cur_mask*obj

    # Make sure it does not jerk too much
    full_mask = cleanup_mask(full_mask, 1, ann_frame_idx)
    
    if 'full_mask' in viewer.layers:
        viewer.layers['full_mask'].data = full_mask
    else:
        viewer.add_labels(full_mask, name='full_mask')
    viewer.layers.selection = {viewer.layers['full_mask']}
    viewer.layers['Shapes'].visible=False

In [14]:
@viewer.bind_key('Shift-A', overwrite=True)
def keep_image(viewer):
    current_mask = viewer.layers['full_mask'].data
    # Keep only the largest contiguous object
    current_mask = keep_largest(current_mask)
    if 'master_labels' in viewer.layers:
        viewer.layers['master_labels'].data[current_mask>0] = np.max(viewer.layers['master_labels'].data) + 1
    else:
        viewer.add_labels(current_mask, name='master_labels')
    print('Visible')
    viewer.layers['Shapes'].visible=True
    print('Selection')
    viewer.layers.selection = {viewer.layers['Shapes']}
    print('Data')
    viewer.layers['Shapes'].data = [dummy_shape]
    print('Visible')
    viewer.layers['Shapes'].visible=False
    viewer.layers['Shapes'].visible=True
    print('Made it')


In [16]:
ski.io.imsave(fname.replace('.tif', '_mask.tif'), viewer.layers['master_labels'].data)

C:\Users\smc\AppData\Local\Temp\ipykernel_23048\283268250.py:1: UserWarning: S:/micro/mg2/ak2718/20250418_DragonFly_mck/2025-04-18/ncol5_488_ncol1_546_dapi_ncol4_647_s06_mask.tif is a low contrast image
  ski.io.imsave(fname.replace('.tif', '_mask.tif'), viewer.layers['master_labels'].data)
